# Qwen3-8B 우선순위 2: 좁고 강한 MBR 실험 노트북

서로 성격이 비슷한 상위권 제출본만 묶어서 **좁은 후보군 MBR**을 수행합니다.

생성 파일:
- `submit_23_narrow_mbr_best2.csv`
- `submit_24_narrow_mbr_best3_clean.csv`
- `submit_25_narrow_mbr_best3_consensus.csv`

핵심 아이디어:
1. 최고 단일 프롬프트 `short_clean_v3`를 기준으로 둠
2. 같은 축의 상위 후보만 넣어서 과한 다양성 제거
3. tie-break는 `short_clean_v3` 쪽에 우선권을 둠


In [ ]:
import sys, pandas as pd
print('python', sys.version)
print('pandas', pd.__version__)


In [ ]:
# -*- coding: utf-8 -*-
from pathlib import Path
import re
import numpy as np
import pandas as pd
from rouge import Rouge

# ============================================================
# CONFIG
# ============================================================
MANUAL_DIR = Path("/root/upstage-nlp-nlp/code/prediction/qwen3_response_only_best_strategy_8b/manual_submissions")

CANDIDATES = {
    "short_clean_v3": MANUAL_DIR / "submit_12_top1_absShort_clean_v3.csv",
    "abstract_short": MANUAL_DIR / "submit_05_top1_abstract_short.csv",
    "mbr_clean_refine": MANUAL_DIR / "submit_16_top1_mbr_clean_refine.csv",
    "mbr_short_best3": MANUAL_DIR / "submit_13_top1_mbr_absShort_best3.csv",
}

OUT_FILES = {
    "narrow_mbr_best2": MANUAL_DIR / "submit_23_narrow_mbr_best2.csv",
    "narrow_mbr_best3_clean": MANUAL_DIR / "submit_24_narrow_mbr_best3_clean.csv",
    "narrow_mbr_best3_consensus": MANUAL_DIR / "submit_25_narrow_mbr_best3_consensus.csv",
}

ROUGE = Rouge()

# ============================================================
# 유틸
# ============================================================
def normalize_text(text: str) -> str:
    text = str(text)
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    text = re.sub(r"<\|.*?\|>", "", text)
    text = re.sub(r"<[^>]+>", "", text)
    text = text.replace("/no_think", " ")
    text = re.sub(r"^요약\s*:\s*", "", text).strip()
    text = re.sub(r"#\s*Person\s*(\d+)\s*#", r"#Person\1#", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text if text else "빈 요약"


def load_submission(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"missing candidate csv: {path}")
    df = pd.read_csv(path)
    if "fname" not in df.columns or "summary" not in df.columns:
        raise ValueError(f"invalid csv format: {path}")
    df = df[["fname", "summary"]].copy()
    df["summary"] = df["summary"].map(normalize_text)
    return df


def pairwise_rouge1_agreement(candidates):
    scores = []
    for j, cj in enumerate(candidates):
        total, cnt = 0.0, 0
        for k, ck in enumerate(candidates):
            if j == k:
                continue
            try:
                s = ROUGE.get_scores([normalize_text(cj)], [normalize_text(ck)])[0]
                total += s["rouge-1"]["f"]
                cnt += 1
            except Exception:
                pass
        scores.append(total / max(cnt, 1))
    return scores


def mbr_ensemble(df_map: dict, preferred_order: list):
    n = len(next(iter(df_map.values())))
    outputs = []
    selected = {name: 0 for name in preferred_order}

    for i in range(n):
        candidates = [df_map[name].iloc[i]["summary"] for name in preferred_order]
        agreements = pairwise_rouge1_agreement(candidates)
        max_score = max(agreements)
        best_indices = [idx for idx, score in enumerate(agreements) if abs(score - max_score) < 1e-12]
        best_idx = best_indices[0]  # preferred_order 앞쪽 후보에 tie-break 우선권
        best_name = preferred_order[best_idx]
        selected[best_name] += 1
        outputs.append(candidates[best_idx])

    return outputs, selected


def save_submission(base_df: pd.DataFrame, summaries, out_path: Path):
    out = pd.DataFrame({"fname": base_df["fname"], "summary": summaries})
    out.to_csv(out_path, index=False)
    print("saved:", out_path)


# ============================================================
# 로딩
# ============================================================
loaded = {name: load_submission(path) for name, path in CANDIDATES.items()}
base = loaded["short_clean_v3"][["fname"]].copy()
for name, df in loaded.items():
    if len(df) != len(base):
        raise ValueError(f"length mismatch: {name}")
    if df["fname"].tolist() != base["fname"].tolist():
        raise ValueError(f"fname order mismatch: {name}")

# ============================================================
# 좁은 강후보 MBR 3종
# ============================================================
plans = {
    "narrow_mbr_best2": ["short_clean_v3", "abstract_short"],
    "narrow_mbr_best3_clean": ["short_clean_v3", "abstract_short", "mbr_clean_refine"],
    "narrow_mbr_best3_consensus": ["short_clean_v3", "abstract_short", "mbr_short_best3"],
}

previews = []
for name, order in plans.items():
    preds, selected = mbr_ensemble(loaded, order)
    save_submission(base, preds, OUT_FILES[name])
    print(name, "selected:", selected)
    previews.append(pd.DataFrame({
        "submission": [name] * 6,
        "fname": base["fname"].head(6),
        "summary": preds[:6],
    }))

print("\n===== preview =====")
print(pd.concat(previews, ignore_index=True))
